# Microduck RL — 세션 2: 걷기 속도 개선 (Kaggle 무료 T4)

- 공개 v8(`velstand/0045i2si_2749`)에서 **warm start**(`MICRODUCK_WARM_START=1`: 가중치/정규화/옵티마이저 유지, 커리큘럼은 0부터 다시)로 시작.
- 과제: **`Mjlab-VelStand-Flat-Backlash-MicroDuck`** (평지 + 기어 백래시). v4(Rough)에서는 지형 커리큘럼이 2.39 → 0으로 무너졌음.
- **`MICRODUCK_SPEED_STAGE=1`** (아래 패치, microduck_rl `bae39f5` 위에 적용):
  - 걷기 선생님(walk anchor, `alpha_walking`)에게 '더 빠른 속도 명령'을 넣어 줘서, 학생이 명령한 속도대로 걷는 시범을 보게 함 (선생님이 명령의 약 0.4–0.5배로만 걷는 게 v8이 느린 원인).
  - 선속도 추종 보상 2 → 3, 명령 범위 앞으로 치우치게 (x −0.3~0.4, y ±0.2).
  - 밀치기·넘어진 자세 스폰·일어나기 경제 등 어려운 조건은 0에서 다시 시작해 2배 느리게 올림 (밀치기 최대는 ~1600회, 엎드린 시작 45%는 ~2800회).
- wandb 없음, 비용 0원. `TIME_BUDGET_H`에서 스스로 멈추고 체크포인트(50회마다) + ONNX를 출력 폴더에 저장.


In [ ]:
# ==== 설정 ====
MODE = "velstand_speed"         # "velstand_speed" | "velstand_resume" | "velocity_flat"
NUM_ENVS = 2048                 # T4(15GB)에서 4096은 메모리 부족. 실패하면 자동으로 1024로 재시도
MAX_ITERATIONS = 6000           # warm start라 0부터 셈. Rough에서 T4 2048env 약 9.2초/반복(평지는 더 빠를 수 있음). 시간 예산이 먼저 끊음
SAVE_INTERVAL = 50
TIME_BUDGET_H = 10.0            # 주간 무료 GPU 30h 중 남은 ~19h 안에서, 12시간 세션 한도보다 여유 있게
RESUME_FROM = None              # None = 공개 v8 (2749). v4의 model_6850은 지형 커리큘럼 붕괴/회전 오차 증가로 쓰지 않음
REPO = "https://github.com/pollen-robotics/microduck_rl.git"
BRANCH = "develop"
COMMIT = "bae39f5"              # CPU에서 v8 번들 이어 학습이 검증된 커밋. None이면 최신 developSNAPSHOT_EVERY_MIN = 30        # 학습 중에도 최신 체크포인트를 출력 폴더에 복사 (세션이 끊겨도 남도록)


In [ ]:
import os, pathlib, subprocess, sys
OUT = pathlib.Path("/kaggle/working/out") if os.path.isdir("/kaggle/working") else pathlib.Path("/content/out")
OUT.mkdir(parents=True, exist_ok=True)
WORK = pathlib.Path("/kaggle/temp" if os.path.isdir("/kaggle") else "/content") / "work"
WORK.mkdir(parents=True, exist_ok=True)
print("output dir:", OUT, "| work dir:", WORK)
!nvidia-smi

GPU 이름이 **Tesla T4**인지, Driver/CUDA Version이 12.4 이상인지 확인하세요 (mujoco_warp는 CUDA 12.4 미만 드라이버에서 `Minimum supported CUDA version: 12.4` 오류를 냄).

In [ ]:
!pip -q install uv
!cd {WORK} && (test -d microduck_rl || git clone -b {BRANCH} {REPO} microduck_rl)
if COMMIT:
    !cd {WORK}/microduck_rl && git checkout -q {COMMIT}
!cd {WORK}/microduck_rl && git log --oneline -1 && uv sync 2>&1 | tail -3

In [ ]:
# SPEED_STAGE 패치 적용 (microduck_rl bae39f5 기준)
PATCH = r'''diff --git a/src/mjlab_microduck/tasks/distill.py b/src/mjlab_microduck/tasks/distill.py
index c0156d3..9e23494 100644
--- a/src/mjlab_microduck/tasks/distill.py
+++ b/src/mjlab_microduck/tasks/distill.py
@@ -108,6 +108,11 @@ def default_bc_cfg() -> dict:
         # |wz| knots, "pos"/"neg": fed |wz| per (lin, target)}. A pure function of the obs,
         # so labels stay consistent; zero yaw (standing, straight walking) is unchanged.
         "anchor_yaw_remap": None,
+        # Teacher-side forward-speed remap (2026-10-29, velstand SPEED_STAGE): same idea for
+        # vx — the anchor walks ~0.4-0.5x the commanded speed, so it is fed the |vx| that
+        # makes it reach the student's command: {"targets": |vx| knots, "fed": fed |vx|}.
+        # Applied after the yaw remap (which reads the TRUE |v_xy|). None = off.
+        "anchor_lin_remap": None,
         "body_active_axes": (2, 3, 4),
         "body_no_teacher_axes": (2,),
         # Flat actor-obs columns ZEROED in the stand expert's input on fallen frames (tilt >
@@ -209,6 +214,18 @@ def remap_anchor_yaw(obs: torch.Tensor, twist_slice: tuple[int, int], table: dic
     return out
 
 
+def remap_anchor_lin(obs: torch.Tensor, twist_slice: tuple[int, int], table: dict) -> torch.Tensor:
+    """Copy of the flat obs with the forward command vx replaced by the one the walk anchor
+    needs to actually walk at that speed (piecewise linear in |vx|, sign kept, 0 → 0)."""
+    out = obs.clone()
+    vx = obs[:, twist_slice[0]]
+    tk = torch.as_tensor([0.0, *table["targets"]], dtype=obs.dtype, device=obs.device)
+    fk = torch.as_tensor([0.0, *table["fed"]], dtype=obs.dtype, device=obs.device)
+    fed = _interp_rows(vx.abs(), tk, fk.expand(vx.numel(), -1))
+    out[:, twist_slice[0]] = torch.sign(vx) * fed
+    return out
+
+
 def _with_flat(td: TensorDict, groups: list[str], flat: torch.Tensor) -> TensorDict:
     """Rebuild the obs groups of ``td`` from a modified flat obs."""
     off = 0
@@ -328,6 +345,11 @@ class PpoWithExpertBc(PPO):
                     anc_flat = remap_anchor_yaw(torch.cat([anc_td[g] for g in groups], dim=-1), tuple(cfg["twist_slice"]),
                                                 cfg["anchor_yaw_remap"])
                     anc_td = _with_flat(anc_td, groups, anc_flat)
+                if cfg.get("anchor_lin_remap"):
+                    anc_td = anc_td.clone()
+                    anc_flat = remap_anchor_lin(torch.cat([anc_td[g] for g in groups], dim=-1), tuple(cfg["twist_slice"]),
+                                                cfg["anchor_lin_remap"])
+                    anc_td = _with_flat(anc_td, groups, anc_flat)
                 target[~fallen] = self.anchor(anc_td)
         n = idx.numel()
         mb = max(cfg.get("min_mini_batch", 1), n // cfg["mini_batches"])
diff --git a/src/mjlab_microduck/tasks/microduck_velstand_env_cfg.py b/src/mjlab_microduck/tasks/microduck_velstand_env_cfg.py
index ff5b6da..3a79e9b 100644
--- a/src/mjlab_microduck/tasks/microduck_velstand_env_cfg.py
+++ b/src/mjlab_microduck/tasks/microduck_velstand_env_cfg.py
@@ -216,6 +216,7 @@ Phases (as before, but with a recovery backstop):
 """
 
 import math
+import os
 
 from mjlab.envs import ManagerBasedRlEnvCfg
 from mjlab.managers import (
@@ -247,6 +248,39 @@ from mjlab_microduck.tasks.symmetry import PpoWithSymmetryCfg
 
 NUM_STEPS_PER_ENV = 24
 
+# ── Speed stage (2026-10-29, free Kaggle session 2; opt-in, MICRODUCK_SPEED_STAGE=1) ──
+# Problem: v8 (0045i2si@2749) walks ~0.17 m/s when commanded 0.4 (~0.09 for 0.2), and a
+# 4127-iteration plain resume on Rough (free run v4) did not change that. Cause: the WALK
+# ANCHOR (alpha_walking 441tzs6d@3750) has the same curve and coef-1.0 BC on every upright
+# frame copies it (same mechanism as the yaw dead zone fixed by ENABLE_ANCHOR_YAW_REMAP).
+# Measured on the anchor itself (CPU probe, Flat-Backlash, no pushes, 2-3 envs per value,
+# mean body vx over 2-6 s): fed vx 0.2 / 0.3 / 0.4 / 0.5 / 0.6 / 0.8 / 0.9 / 1.0 / 1.2 →
+# 0.10 / 0.15 / 0.20 / 0.22 / 0.28 / 0.31 / 0.35 / 0.37 / 0.45 m/s, 0 falls up to 1.0,
+# 33 % tilted at 1.2; backward ≈ the same magnitude (-0.4 → -0.20, -0.8 → -0.34).
+# So, behind this flag:
+#   - ANCHOR LIN REMAP: the anchor is fed the forward command that makes IT walk at the
+#     student's commanded |vx| (monotone smoothed inverse of the probe, capped at 1.0 =
+#     no-fall region). Student obs keep the true command; standing (0) and vy unchanged.
+#   - track_linear_velocity weight 2 → 3 so PPO's reward agrees with the faster labels.
+#   - forward-biased x range (-0.3, 0.4) (was ±0.4), lateral ±0.2 (was ±0.3).
+#   - curricula are meant to RESTART (launch with MICRODUCK_WARM_START=1) and velstand's
+#     own ramps (fell_over off, crouch/prone/wide spawns, topple pushes, recovery economics,
+#     protection costs) are stretched ×SPEED_RAMP_STRETCH, so the first ~200 iterations are
+#     clean walking with only the mild ±0.3 m/s stumble pushes, and full get-up/topple
+#     difficulty only returns at ~1600 (pushes) / ~2800 (45 % prone) iterations.
+#   - use with the FLAT task (Mjlab-VelStand-Flat-Backlash-MicroDuck): run v4's terrain
+#     curriculum collapsed 2.39 → ~0 on Rough.
+SPEED_STAGE = os.environ.get("MICRODUCK_SPEED_STAGE", "") not in ("", "0")
+SPEED_RAMP_STRETCH = 2 if SPEED_STAGE else 1
+SPEED_TRACK_LIN_WEIGHT = 3.0
+SPEED_LIN_VEL_X = (-0.3, 0.4)
+SPEED_LIN_VEL_Y = (-0.2, 0.2)
+ANCHOR_LIN_REMAP = {
+    # target |vx| knots (m/s) → fed |vx| for the walk anchor; 0 → 0, clamped beyond the last knot
+    "targets": (0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40),
+    "fed":     (0.20, 0.30, 0.40, 0.55, 0.70, 0.90, 1.00),
+}
+
 # Warm start from the deployed walk (see module docstring). Collapses every
 # velocity-recipe curriculum to its final stage so the loaded walk is trained
 # under the conditions it was trained under. Set False to train from scratch
@@ -258,7 +292,7 @@ WARM_START_CHECKPOINT = "model_3750.pt"
 # Phase boundaries (PPO iterations; env step counter scales by num_steps_per_env=24).
 # Warm start: the walk exists at iter 0, so fell_over only needs a short
 # adaptation window to the all-collisions model before falls become data.
-FELL_OVER_DISABLE_ITER = 100 if WARM_START else 500
+FELL_OVER_DISABLE_ITER = (100 if WARM_START else 500) * SPEED_RAMP_STRETCH
 
 # Fallen gates. LESSON (first rebase training run): the recovery REWARDS must
 # gate on TILT ONLY. Gating them on low height too made SITTING (z≈0.07, trunk
@@ -296,7 +330,7 @@ RECOVERED_UP_Z = 0.09
 # about the walk; it bought a TAX-FREE window (fell_over off at 500 → econ on
 # at 1200) where natural-fall get-up attempts cost nothing and the dense
 # progress terms alone could teach them. Run-7 restores it.
-RECOVERY_ECON_KICKIN_ITER = 600 if WARM_START else 1200
+RECOVERY_ECON_KICKIN_ITER = (600 if WARM_START else 1200) * SPEED_RAMP_STRETCH
 
 # Run-2 fix (1bqctpkq, 0 recoveries @1085 even with the attempt tax relieved):
 # distill the deployed stand expert into the fallen frames (see distill.py).
@@ -457,7 +491,7 @@ FALLEN_SMOOTHNESS_SCALE = 0.1
 
 # Servo-protection costs ramp (see docstring). 25% from step 0 keeps the
 # gradient alive; full weight after the recovery economics are in place.
-PROTECT_FULL_ITER = RECOVERY_ECON_KICKIN_ITER + 400
+PROTECT_FULL_ITER = RECOVERY_ECON_KICKIN_ITER + 400 * SPEED_RAMP_STRETCH
 PROTECT_STAGE0_FRAC = 0.25
 SERVO_IMPACT_WEIGHT = -0.02     # per N above 2 N on servo housings, per step
 HEAD_IMPACT_WEIGHT = -0.01      # per N above 15 N on the head subtree
@@ -479,8 +513,8 @@ TOPPLE_PUSH_INTERVAL_S = (4.0, 8.0)
 TOPPLE_PUSH_STAGES = [
     {"step": 0,                                          "velocity_range": {"x": (-0.3, 0.3), "y": (-0.3, 0.3)}},
     {"step": FELL_OVER_DISABLE_ITER * NUM_STEPS_PER_ENV, "velocity_range": {"x": (-0.6, 0.6), "y": (-0.6, 0.6)}},
-    {"step": 400 * NUM_STEPS_PER_ENV,                    "velocity_range": {"x": (-0.9, 0.9), "y": (-0.9, 0.9)}},
-    {"step": 800 * NUM_STEPS_PER_ENV,                    "velocity_range": {"x": (-1.2, 1.2), "y": (-1.2, 1.2)}},
+    {"step": 400 * SPEED_RAMP_STRETCH * NUM_STEPS_PER_ENV, "velocity_range": {"x": (-0.9, 0.9), "y": (-0.9, 0.9)}},
+    {"step": 800 * SPEED_RAMP_STRETCH * NUM_STEPS_PER_ENV, "velocity_range": {"x": (-1.2, 1.2), "y": (-1.2, 1.2)}},
 ]
 
 # Failed-recovery backstop: continuously fallen this long → terminate/reset.
@@ -503,7 +537,7 @@ FALLEN_TIMEOUT_S = 8.0
 # fix (2): side_prob puts that fraction of the prone slice ON A SIDE (the
 # dominant natural fall end-state, 79% in eval); the rest splits by face_down_prob.
 # Crouch slice 0.15 → 0.20 and from iter 150: it doubles as stand-tall data.
-_PRONE_ITERS = (150, 700, 1000, 1400) if WARM_START else (800, 1500, 2000, 2500)
+_PRONE_ITERS = tuple(i * SPEED_RAMP_STRETCH for i in ((150, 700, 1000, 1400) if WARM_START else (800, 1500, 2000, 2500)))
 PRONE_SIDE_PROB = 0.5
 PRONE_RAMP_STAGES = [
     {"step": 0,                                   "params": {"prone_prob": 0.00, "face_down_prob": 1.0,  "side_prob": PRONE_SIDE_PROB, "crouch_prob": 0.00}},
@@ -630,6 +664,12 @@ def make_microduck_velstand_env_cfg(play: bool = False, rough: bool = False) ->
             params={"std": TRACK_YAW_FINE_STD, "command_name": cfg.rewards["track_angular_velocity"].params["command_name"]},
         )
 
+    if SPEED_STAGE:
+        tw = cfg.commands["twist"]
+        tw.ranges.lin_vel_x = SPEED_LIN_VEL_X
+        tw.ranges.lin_vel_y = SPEED_LIN_VEL_Y
+        cfg.rewards["track_linear_velocity"].weight = SPEED_TRACK_LIN_WEIGHT
+
     # True full-collision model: the robot can lie on / push off any part, and
     # the servo housings are named so the impact sensor below can single them
     # out. 70 collision geoms (vs 11) → more simultaneous contacts in a pile-up;
@@ -993,6 +1033,7 @@ MicroduckVelStandRlCfg = RslRlOnPolicyRunnerCfg(
             **({"body_slice": (55, 61)} if ENABLE_BODY_CONTROL else {}),
             **({"unanchor_turn_in_place": True} if ENABLE_YAW_FIX else {}),
             **({"anchor_yaw_remap": ANCHOR_YAW_REMAP} if ENABLE_ANCHOR_YAW_REMAP else {}),
+            **({"anchor_lin_remap": ANCHOR_LIN_REMAP} if SPEED_STAGE else {}),
             **({"expert_mask_cols": TEACHER_MASK_COLS, "expert_mask_tilt_deg": TEACHER_MASK_TILT_DEG} if ENABLE_JOINT_MASKED_TEACHER else {}),
         } if ENABLE_EXPERT_BC else None,
     ),
'''
(WORK / 'speed_stage.patch').write_text(PATCH)
(OUT / 'speed_stage.patch').write_text(PATCH)
if MODE == 'velstand_speed':
    !cd {WORK}/microduck_rl && (git apply --check {WORK}/speed_stage.patch && git apply {WORK}/speed_stage.patch || git apply -R --check {WORK}/speed_stage.patch && echo already applied) && git diff --stat


In [ ]:
# GPU 인식 확인 (torch + warp)
!cd {WORK}/microduck_rl && uv run --no-sync python -c "import torch,warp as wp; print('torch', torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-'); wp.init(); print('warp devices', wp.get_devices())"

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!cd {WORK}/microduck_rl && uv run --no-sync python -c "import torch,warp as wp,mujoco; print('torch',torch.__version__,'cuda',torch.version.cuda,'cap',torch.cuda.get_device_capability(0)); print('warp',wp.__version__,'mujoco',mujoco.__version__)"

In [ ]:
import shlex, time
env = dict(os.environ, WANDB_MODE="disabled", CUDA_VISIBLE_DEVICES="0")
if MODE == "velstand_speed":
    task = "Mjlab-VelStand-Flat-Backlash-MicroDuck"
    env.update(MICRODUCK_SPEED_STAGE="1", MICRODUCK_WARM_START="1")
    init = RESUME_FROM or "hf://pollen-robotics/microduck-training/bundles/velstand/0045i2si_2749"
    extra = ["--init-checkpoint", init]
elif MODE == "velstand_resume":
    task = "Mjlab-VelStand-Rough-Backlash-MicroDuck"
    init = RESUME_FROM or "hf://pollen-robotics/microduck-training/bundles/velstand/0045i2si_2749"
    extra = ["--init-checkpoint", init]
elif MODE == "velocity_flat":
    task = "Mjlab-Velocity-Flat-MicroDuck"
    extra = ["--init-checkpoint", RESUME_FROM] if RESUME_FROM else []
else:
    raise ValueError(MODE)
def build_cmd(num_envs):
  return ["uv", "run", "--no-sync", "train", task,
         "--agent.logger", "tensorboard",
         "--env.scene.num-envs", str(num_envs),
         "--agent.max-iterations", str(MAX_ITERATIONS),
         "--agent.save-interval", str(SAVE_INTERVAL),
         "--agent.run-name", f"free_{MODE}"] + extra
import glob, re, shutil
def snapshot():
    pts = glob.glob(str(WORK / "microduck_rl/logs/rsl_rl/*/*/model_*.pt"))
    if pts:
        last = max(pts, key=os.path.getmtime)
        for old in glob.glob(str(OUT / "snapshot_model_*.pt")): os.remove(old)
        shutil.copy(last, OUT / ("snapshot_" + os.path.basename(last)))
t0 = time.time(); t_snap = t0
for n_env in [NUM_ENVS, 1024, 512]:
    cmd = build_cmd(n_env)
    print(" ".join(shlex.quote(c) for c in cmd))
    log = open(OUT / "train.log", "a"); log.write(f"=== try num_envs={n_env}\n")
    p = subprocess.Popen(cmd, cwd=WORK / "microduck_rl", env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in p.stdout:
            log.write(line); log.flush()
            if any(k in line for k in ("Learning iteration", "Steps per second", "Mean reward", "Iteration time", "Error", "Traceback")):
                print(line.rstrip())
            if time.time() - t_snap > SNAPSHOT_EVERY_MIN * 60:
                snapshot(); t_snap = time.time()
            if time.time() - t0 > TIME_BUDGET_H * 3600:
                print("time budget reached, stopping"); p.terminate(); break
    finally:
        p.wait(); log.close()
    print("exit", p.returncode, "elapsed h", round((time.time() - t0) / 3600, 2))
    oom = "Failed to allocate" in open(OUT / "train.log").read().split("=== try")[-1]
    if p.returncode == 0 or not oom: break
    print("OOM at", n_env, "envs -> retry smaller")


In [ ]:
import re, statistics
txt=open(OUT/'train.log').read()
sps=[float(x) for x in re.findall(r'Steps per second:\s*([\d.]+)',txt)]
it=[float(x) for x in re.findall(r'Iteration time:\s*([\d.]+)s',txt)]
print('iterations logged',len(it))
if sps: print('steps/s median',statistics.median(sps),'last',sps[-1])
if it: print('iter time median',statistics.median(it),'s')
print(txt[-3000:])

In [ ]:
# 최신 체크포인트를 ONNX로 내보내고 출력 폴더에 복사
import glob, re, shutil
runs = sorted(glob.glob(str(WORK / "microduck_rl/logs/rsl_rl/*/*")), key=os.path.getmtime)
run = runs[-1]; print("run dir:", run)
pts = sorted(glob.glob(run + "/model_*.pt"), key=lambda f: int(re.findall(r"model_(\d+)", f)[0]))
last = pts[-1]; print("last checkpoint:", last)
onnx = OUT / (pathlib.Path(run).name + "_" + pathlib.Path(last).stem + ".onnx")
subprocess.run(["uv", "run", "--no-sync", "python", "scripts/export.py", task,
                "--checkpoint-file", last, "--onnx-file", str(onnx)],
               cwd=WORK / "microduck_rl", env=env, check=True)
shutil.copy(last, OUT)
for f in glob.glob(run + "/events.out.tfevents*") + glob.glob(run + "/params/*"):
    shutil.copy(f, OUT)
!ls -la {OUT}

결과는 출력 폴더(Kaggle: Output 탭 `/kaggle/working/out`, Colab: 왼쪽 파일 패널 `/content/out`)에 남아요. Kaggle은 **Save Version → Save & Run All (Commit)** 으로 돌리면 브라우저를 닫아도 끝까지 돌고 출력이 보존돼요. 다음 세션에서는 이 출력(`model_N.pt`)을 데이터셋으로 붙이고 `RESUME_FROM`에 경로를 넣으면 이어서 학습돼요.